# Prize-Winning Hackathon Submission

This notebook contains the **prize-winning solution** for the All Cups hackathon challenge "Sales Rhythms".

**Approach:**
- LightGBM (MAE / regression_l1) with sample weights (w=7 for qty>0, w=1 for qty=0)
- 18 core features (calendar, price, promo, leftovers, lags, rolling stats, item statistics)
- Iterative 14-day forecasting (lags are updated day by day)
- Post-processing: clip to [0, prev_leftovers], threshold=1.8, `rint` rounding


In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import os
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
THRESHOLD = 1.8
CALIB = 1.0
ROUNDING = 'rint'
W_POS = 7.0

DATA_DIR = r"C:\Yandex.Disk\Yandex.Disk\ML HSE\All cups"

def wmae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    w = np.where(y_true > 0, 7.0, 1.0)
    return float(np.sum(w * np.abs(y_true - y_pred)) / np.sum(w))

def round_qty(raw, mode):
    if mode == 'floor':
        return np.floor(raw)
    if mode == 'ceil':
        return np.ceil(raw)
    if mode == 'rint':
        return np.rint(raw)
    return np.floor(raw + 0.5)

print('Setup done')

Setup done


## 1. Data Loading

In [2]:
train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
test = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

for df in [train, test, sample]:
    df['dt'] = pd.to_datetime(df['dt'])

train_end = train['dt'].max()
test_ids = test['nm_id'].unique()
train_f = train[train['nm_id'].isin(test_ids)].copy()

print(f'train: {train.shape}, test: {test.shape}, sample: {sample.shape}')
print(f'train_end: {train_end.date()}, test items: {len(test_ids)}')

train: (309648, 6), test: (12856, 5), sample: (12856, 3)
train_end: 2025-07-07, test items: 949


## 2. Build Panel (all items x all days)

In [3]:
all_days = pd.date_range(train['dt'].min(), test['dt'].max(), freq='D')
panel = pd.MultiIndex.from_product([test_ids, all_days], names=['nm_id','dt']).to_frame(index=False)

full = pd.concat([
    train_f.assign(source='train'),
    test.assign(qty=np.nan, source='test')
], ignore_index=True)

panel = panel.merge(full, on=['nm_id','dt'], how='left')
panel['source'] = panel['source'].fillna('missing')
panel = panel.sort_values(['nm_id','dt']).reset_index(drop=True)

# Заполнение пропусков
panel['is_promo'] = panel['is_promo'].fillna(0).astype('int8')
panel['prev_leftovers'] = panel['prev_leftovers'].fillna(0).astype('float32')
panel['price'] = panel['price'].astype('float32')
panel['price'] = panel.groupby('nm_id')['price'].ffill()
imp_price = train_f.groupby('nm_id')['price'].median()
panel['price'] = panel['price'].fillna(panel['nm_id'].map(imp_price)).fillna(train_f['price'].median())
panel.loc[panel['dt'] <= train_end, 'qty'] = panel.loc[panel['dt'] <= train_end, 'qty'].fillna(0)

print(f'panel: {panel.shape}')

panel: (363467, 7)


## 3. Feature Engineering (18 features)

In [ ]:
# Calendar features
panel['dow'] = panel['dt'].dt.dayofweek.astype('int8')
panel['month'] = panel['dt'].dt.month.astype('int8')
panel['is_weekend'] = (panel['dow'] >= 5).astype('int8')

# ID encoding
le = LabelEncoder()
panel['nm_id_le'] = le.fit_transform(panel['nm_id'])

# Price features
panel['log_price'] = np.log1p(panel['price']).astype('float32')
grp = panel.groupby('nm_id')
panel['price_lag1'] = grp['price'].shift(1).fillna(panel['price'])
panel['price_chg1'] = ((panel['price'] - panel['price_lag1']) / (panel['price_lag1'] + 1)).fillna(0).astype('float32')

# Inventory / leftovers
panel['log_leftovers'] = np.log1p(panel['prev_leftovers']).astype('float32')
panel['is_stockout'] = (panel['prev_leftovers'] <= 0).astype('int8')

# Technical flags
panel['row_missing'] = (panel['source'] == 'missing').astype('int8')

# Item-level statistics (from train)
item_pos_rate = train_f.groupby('nm_id')['qty'].apply(lambda x: (x > 0).mean())
item_mean_qty = train_f.groupby('nm_id')['qty'].mean()
item_mean_pos = train_f[train_f['qty'] > 0].groupby('nm_id')['qty'].mean()
panel['item_pos_rate'] = panel['nm_id'].map(item_pos_rate).fillna(0).astype('float32')
panel['item_mean_qty'] = panel['nm_id'].map(item_mean_qty).fillna(0).astype('float32')
panel['item_mean_pos'] = panel['nm_id'].map(item_mean_pos).fillna(0).astype('float32')

# Lags and rolling features
grp = panel.groupby('nm_id')
for lag in [1, 7]:
    panel[f'lag_{lag}'] = grp['qty'].shift(lag)
lcols = ['lag_1', 'lag_7']
panel[lcols] = panel.groupby('nm_id')[lcols].ffill().fillna(0)

for w in [7, 14]:
    panel[f'roll_mean_{w}'] = grp['qty'].shift().rolling(w, min_periods=1).mean().fillna(0)

panel['pos_rate_14'] = grp['qty'].shift().gt(0).rolling(14, min_periods=1).mean().fillna(0)

print('Features done')

Features done


In [ ]:
# List of 18 features
FEATURES = [
    'nm_id_le', 'dow', 'month', 'is_weekend',
    'log_price', 'price_chg1', 'is_promo',
    'log_leftovers', 'is_stockout', 'row_missing',
    'item_pos_rate', 'item_mean_qty', 'item_mean_pos',
    'lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_14', 'pos_rate_14'
]
print(f'{len(FEATURES)} features: {FEATURES}')

18 features: ['nm_id_le', 'dow', 'month', 'is_weekend', 'log_price', 'price_chg1', 'is_promo', 'log_leftovers', 'is_stockout', 'row_missing', 'item_pos_rate', 'item_mean_qty', 'item_mean_pos', 'lag_1', 'lag_7', 'roll_mean_7', 'roll_mean_14', 'pos_rate_14']


## 4. LightGBM Training

In [ ]:
# Prepare train/validation split
df = panel[(panel['dt'] <= train_end) & (panel['source'] == 'train')].copy()

val_start = train_end - pd.Timedelta(days=13)
tr = df[df['dt'] < val_start]
va = df[df['dt'] >= val_start]

params = {
    'objective': 'regression_l1',
    'learning_rate': 0.05,
    'num_leaves': 127,
    'min_data_in_leaf': 80,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,
    'lambda_l2': 10.0,
    'feature_pre_filter': False,
    'verbose': -1,
    'seed': SEED,
}

# Train with early stopping to determine best_iteration
model_val = lgb.train(
    params,
    lgb.Dataset(tr[FEATURES], tr['qty'].values,
                weight=np.where(tr['qty'].values > 0, W_POS, 1.), free_raw_data=False),
    num_boost_round=2000,
    valid_sets=[lgb.Dataset(va[FEATURES], va['qty'].values,
                weight=np.where(va['qty'].values > 0, W_POS, 1.), free_raw_data=False)],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)]
)

BEST_ROUND = model_val.best_iteration
print(f'Best iteration: {BEST_ROUND}')

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[70]	valid_0's l1: 2.16154
Best iteration: 70


In [ ]:
# Final model on ALL train data with best_round
model = lgb.train(
    params,
    lgb.Dataset(df[FEATURES], df['qty'].values,
                weight=np.where(df['qty'].values > 0, W_POS, 1.), free_raw_data=False),
    num_boost_round=BEST_ROUND
)

print(f'Final model trained with {BEST_ROUND} rounds on {len(df)} rows')

Final model trained with 70 rounds on 150247 rows


## 5. Iterative Test Forecasting (14 days)

Key idea: for each test day, lag features are recomputed using predictions from previous days.

Post-processing:
- `raw = max(prediction, 0)`
- `raw = raw * CALIB`
- `raw = min(raw, prev_leftovers)` — clip by available leftovers
- If `raw < threshold (1.8)` -> 0, otherwise apply `ROUNDING`

In [ ]:
HD = 60  # stored history length

# Build history for the last HD days before test
hist_start = train_end - pd.Timedelta(days=HD - 1)
hist_data = panel[
    (panel['nm_id'].isin(test_ids)) &
    (panel['dt'] >= hist_start) &
    (panel['dt'] <= train_end)
][['nm_id', 'dt', 'qty']].copy()
hist_data['qty'] = hist_data['qty'].fillna(0)

hm = {
    nm: grp.sort_values('dt')['qty'].values.astype(float).tolist()
    for nm, grp in hist_data.groupby('nm_id')
}
for nm in test_ids:
    if nm not in hm:
        hm[nm] = [0.0] * HD

def lag_feats(hist):
    """Compute lag-based features from history."""
    L = len(hist)
    h = np.array(hist, dtype=np.float32)
    f = {}
    for lag in [1, 7]:
        f[f'lag_{lag}'] = float(h[-lag]) if L >= lag else 0.0
    for w in [7, 14]:
        win = h[-w:] if L >= w else h
        f[f'roll_mean_{w}'] = float(np.mean(win)) if len(win) else 0.0
    win14 = h[-14:] if L >= 14 else h
    f['pos_rate_14'] = float(np.mean(np.array(win14) > 0)) if len(win14) else 0.0
    return f

# Iterative forecasting
preds = []
test_dates = sorted(test['dt'].unique())

for d in test_dates:
    dd = panel[panel['dt'] == d].copy()
    
    # Recompute lag features from up-to-date history
    lfl = [lag_feats(hm.get(nm, [0.0] * HD)) for nm in dd['nm_id'].values]
    ldf = pd.DataFrame(lfl)
    for c in ldf.columns:
        dd[c] = ldf[c].values
    
    # Predict
    raw = np.maximum(model.predict(dd[FEATURES]), 0)
    raw = raw * CALIB
    raw = np.minimum(raw, dd['prev_leftovers'].values)
    
    # Apply threshold and rounding
    pf = np.where(raw >= THRESHOLD, round_qty(raw, ROUNDING), 0).astype(int)
    pf = np.maximum(pf, 0)
    
    out = dd[['nm_id', 'dt']].copy()
    out['qty'] = pf
    preds.append(out)

pred_test = pd.concat(preds, ignore_index=True)
print(f'Predictions: {pred_test.shape}')
print(f'Mean qty: {pred_test["qty"].mean():.4f}')
print(f'Zero rate: {(pred_test["qty"] == 0).mean():.4f}')

Predictions: (13286, 3)
Mean qty: 0.9506
Zero rate: 0.8136


## 6. Build and Save Submission

In [9]:
sub = sample[['nm_id', 'dt']].merge(pred_test, on=['nm_id', 'dt'], how='left')
sub['qty'] = sub['qty'].fillna(0).astype(int)

out_path = os.path.join(DATA_DIR, 'submission_v7_t1.8.csv')
sub.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {sub.shape}')
print(f'Mean qty: {sub["qty"].mean():.4f}')
print(f'Zero rate: {(sub["qty"] == 0).mean():.4f}')
print(f'\nPublic wMAE: 1.787469744926457')
print(f'Public score: 0.359')
print()
sub.describe()

Saved: C:\Yandex.Disk\Yandex.Disk\ML HSE\All cups\submission_v7_t1.8.csv
Shape: (12856, 3)
Mean qty: 0.9824
Zero rate: 0.8074

Public wMAE: 1.787469744926457
Public score: 0.359



,dt,qty
count,12856,12856.000000
mean,2025-07-14 11:57:38.867455,0.982421
min,2025-07-08 00:00:00,0.000000
25%,2025-07-11 00:00:00,0.000000
50%,2025-07-14 00:00:00,0.000000
75%,2025-07-18 00:00:00,0.000000
max,2025-07-21 00:00:00,52.000000
std,NaN,3.344292
